In [25]:
import csv, re, string, time
from dataclasses import dataclass, asdict
from typing import Optional
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.physics.utoronto.ca"
MEMBERS_URL = f"{BASE_URL}/members/"
OUT_FILE = "uoft_physics_faculty.csv"
REQUEST_DELAY = 1.0
FILTER_FACULTY = True          # False = keep everyone (students, staff, ...)

HEADERS = {"User-Agent": "Mozilla/5.0 (research data collection; contact: your-email@example.com)"}

FACULTY_TITLES = [
    "Tenure-Stream Faculty", "Teaching-Stream Faculty", "Emeritus Professor",
    "Cross Appointed", "Adjunct/Status-Only", "Sessional Lecturer", "Visiting Scholar",
]
SITE_NAME_PATTERNS = ("university of toronto", "department of physics")
NON_PROFILE = {"member_types", "department_groups", "page", "feed"}


@dataclass
class FacultyMember:
    name: str = ""
    title: str = ""
    department: str = ""
    email: str = ""
    phone: str = ""
    personal_website: str = ""
    research_group: str = ""
    profile_url: str = ""
    photo_url: str = ""


def get_soup(url: str, retries: int = 3) -> Optional[BeautifulSoup]:
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"  [warn] {attempt}/{retries} {url}: {e}")
            time.sleep(1.5 * attempt)
    return None


def looks_like_site_name(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in SITE_NAME_PATTERNS)


def extract_name(soup, main, url: str) -> str:
    cands = []
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        cands.append(og["content"])
    if soup.title and soup.title.get_text():
        cands.append(soup.title.get_text())
    for c in cands:
        first = re.split(r"\s[|\-–—]\s", c)[0].strip()
        if first and not looks_like_site_name(first):
            return first
    for h in main.find_all(["h1", "h2", "h3"]):
        txt = h.get_text(" ", strip=True)
        if txt and not looks_like_site_name(txt):
            return txt
    slug = urlparse(url).path.rstrip("/").split("/")[-1]
    return slug.replace("-", " ").title()


def is_profile_path(path: str) -> bool:
    parts = [p for p in path.split("/") if p]
    return len(parts) == 2 and parts[0] == "members" and parts[1] not in NON_PROFILE


def links_from_soup(soup, page_url) -> set:
    found = set()
    for a in soup.find_all("a", href=True):
        full = urljoin(page_url, a["href"]).split("#")[0].split("?")[0]
        u = urlparse(full)
        if u.netloc.endswith("physics.utoronto.ca") and is_profile_path(u.path):
            found.add(f"{BASE_URL}{u.path.rstrip('/')}/")
    return found


def collect_member_links() -> list:
    links = set()
    for letter in string.ascii_uppercase:
        soup = get_soup(f"{MEMBERS_URL}?letter={letter}")
        if soup:
            links |= links_from_soup(soup, MEMBERS_URL)
        time.sleep(REQUEST_DELAY)
    print(f"  letter pages -> {len(links)} links")
    if not links:
        for sm in ("/sitemap.xml", "/wp-sitemap.xml", "/sitemap_index.xml"):
            try:
                r = requests.get(BASE_URL + sm, headers=HEADERS, timeout=20)
                if r.ok:
                    for loc in re.findall(r"<loc>(.*?)</loc>", r.text):
                        u = urlparse(loc)
                        if is_profile_path(u.path):
                            links.add(f"{BASE_URL}{u.path.rstrip('/')}/")
            except requests.RequestException:
                pass
        print(f"  sitemap fallback -> {len(links)} links")
    return sorted(links)


def parse_member_page(url: str) -> Optional[FacultyMember]:
    soup = get_soup(url)
    if soup is None:
        return None
    main = soup.select_one("main") or soup
    m = FacultyMember(profile_url=url)
    m.name = extract_name(soup, main, url)            # <-- the fix

    text = main.get_text("\n", strip=True)
    for t in FACULTY_TITLES:
        if t in text:
            m.title = t
            break
    d = re.search(r"(Department of [A-Za-z &,]+)", text)
    if d:
        m.department = d.group(1).strip()

    e = main.select_one("a[href^='mailto:']")
    if e:
        m.email = e["href"].replace("mailto:", "").split("?")[0].strip()
    p = main.select_one("a[href^='tel:']")
    if p:
        m.phone = p.get_text(strip=True) or p["href"].replace("tel:", "")

    for a in main.select("a[href^='http']"):
        href = a["href"]
        if "utoronto.ca" not in href or "~" in href:
            m.personal_website = href
            break

    g = main.select_one("a[href*='/members/department_groups/']")
    if g:
        m.research_group = g.get_text(strip=True)
    img = main.select_one("img[src*='/pcsapps/media/people/']")
    if img:
        m.photo_url = urljoin(BASE_URL, img["src"])
    return m


def save_to_csv(members, filename=OUT_FILE):
    if not members:
        print("No data to save.")
        return
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(asdict(members[0]).keys()))
        w.writeheader()
        for m in members:
            w.writerow(asdict(m))
    print(f"{len(members)} records saved to '{filename}'.")


def main():
    print("[1] Collecting profile links ...")
    links = collect_member_links()
    print(f"Total profile links: {len(links)}")
    if not links:
        print("!! 0 links. Run the diagnostic cell to see what the server returns.")
        return

    print("[2] Parsing profiles ...")
    rows, skipped = [], 0
    for i, link in enumerate(links, 1):
        m = parse_member_page(link)
        if m:
            if FILTER_FACULTY and m.title not in FACULTY_TITLES:
                skipped += 1
            else:
                rows.append(m)
        if i % 20 == 0:
            print(f"  {i}/{len(links)}")
        time.sleep(REQUEST_DELAY)
    print(f"Kept {len(rows)}, filtered out {skipped} (non-faculty / no title found)")
    save_to_csv(rows)


main()

[1] Collecting profile links ...
  letter pages -> 569 links
Total profile links: 569
[2] Parsing profiles ...
  20/569
  40/569
  60/569
  80/569
  100/569
  120/569
  140/569
  160/569
  180/569
  200/569
  220/569
  240/569
  260/569
  280/569
  300/569
  320/569
  340/569
  360/569
  380/569
  400/569
  420/569
  440/569
  460/569
  480/569
  500/569
  520/569
  540/569
  560/569
Kept 134, filtered out 435 (non-faculty / no title found)
134 records saved to 'uoft_physics_faculty.csv'.


In [26]:
import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 45)
df = pd.read_csv("uoft_physics_faculty.csv")
print(df.shape)
df.head(10)

(134, 9)


,name,title,department,email,phone,personal_website,research_group,profile_url,photo_url
0,Abbatt Jonathan,Cross Appointed,Department of Chemistry,jabbatt@chem.utoronto.ca,NaN,https://www.abbattgroup.ca/,"EAPP - Earth, Atmospheric and Planetary P...",https://www.physics.utoronto.ca/members/a...,NaN
1,Bailey David C,Emeritus Professor,Department of Physics,david.bailey@utoronto.ca,416-978-4993,http://www.physics.utoronto.ca/~dbailey/,EHEP - Experimental Particle Physics and ...,https://www.physics.utoronto.ca/members/b...,https://www.physics.utoronto.ca/pcsapps/m...
2,Bailey Richard C,Emeritus Professor,Department of Physics,r.bailey@utoronto.ca,416-978-3231,http://www.physics.utoronto.ca/~bailey/,"EAPP - Earth, Atmospheric and Planetary P...",https://www.physics.utoronto.ca/members/b...,https://www.physics.utoronto.ca/pcsapps/m...
3,Barzda Virginijus,Tenure-Stream Faculty,Department of Chemical and Physical Sciences,virgis.barzda@utoronto.ca,905 828-3821,NaN,BIOPHY - Biological Physics,https://www.physics.utoronto.ca/members/b...,https://www.physics.utoronto.ca/pcsapps/m...
4,Bond J Richard,Cross Appointed,NaN,bond@cita.utoronto.ca,416-978-6874,https://www.cita.utoronto.ca/~bond/,THEP - Theoretical High Energy Physics,https://www.physics.utoronto.ca/members/b...,https://www.physics.utoronto.ca/pcsapps/m...
5,Braverman Boris,Tenure-Stream Faculty,Department of Physics,boris.braverman@utoronto.ca,"416-978-4395, 416-978-2948",NaN,QO - Quantum Optics,https://www.physics.utoronto.ca/members/b...,https://www.physics.utoronto.ca/pcsapps/m...
6,Chun Kin Yip,Emeritus Professor,Department of Physics,chun@physics.utoronto.ca,416-978-1499,http://www.physics.utoronto.ca/~chun/,NaN,https://www.physics.utoronto.ca/members/c...,https://www.physics.utoronto.ca/pcsapps/m...
7,Code R Fraser,Emeritus Professor,Department of Chemical and Physical Sciences,code@physics.utoronto.ca,416-978-5220,http://www.physics.utoronto.ca/~code/,BIOPHY - Biological Physics,https://www.physics.utoronto.ca/members/c...,https://www.physics.utoronto.ca/pcsapps/m...
8,Crepel Valentin,Tenure-Stream Faculty,Department of Physics,valentin.crepel@utoronto.ca,416-946-3735,NaN,QCMP - Quantum Condensed Matter Physics,https://www.physics.utoronto.ca/members/c...,https://www.physics.utoronto.ca/pcsapps/m...
9,Curtin David,Tenure-Stream Faculty,Department of Physics,d.curtin@utoronto.ca,416-978-4784,NaN,THEP - Theoretical High Energy Physics,https://www.physics.utoronto.ca/members/c...,https://www.physics.utoronto.ca/pcsapps/m...


In [27]:
r = requests.get("https://www.physics.utoronto.ca/members/?letter=B", headers=HEADERS, timeout=20)
print(r.status_code, len(r.text))
s = BeautifulSoup(r.text, "html.parser")
hrefs = [a["href"] for a in s.find_all("a", href=True) if "members" in a["href"]]
print(len(hrefs)); print(hrefs[:30])
print([h.get_text(strip=True) for h in s.find_all(["h1","h2","h3"])][:10])

200 127541
40
['/members/', '/members/', '/members/badbaria-manthan/', '/members/bailey-david-c/', '/members/bailey-richard-c/', '/members/baker-noah/', '/members/balaji-adithi/', '/members/balogun-olalekan/', '/members/bamba-krishang/', '/members/banwait-parveer/', '/members/bao-zhuoran/', '/members/barbieri-andrea/', '/members/barzda-virginijus/', '/members/basu-jahnabi/', '/members/bates-darby/', '/members/beadle-carl/', '/members/beauchesne-blanchet-etienne/', '/members/bekele-bereket-ngussie/', '/members/bera-sandip/', '/members/berthiaume-raphael/', '/members/bhullar-mandev/', '/members/biel-krystyna/', '/members/binoy-brianna/', '/members/biske-spencer/', '/members/bojanich-alex/', '/members/bond-j-richard/', '/members/bookatz-gidon/', '/members/botelho-guilherme/', '/members/brandstatter-florian-christian/', '/members/braverman-boris/']
['Department of Physics', 'Members', 'Results35']
